# Assignment 1A — Data poisoning and activation clustering

Create two attacks on a digit classifier, then apply activation clustering to defend against each one.

| Section | Task | Points |
|---|---|---:|
| 1 | Synthetic lint-trigger attack | 30 |
| 2 | Natural-feature attack using 1s with hats | 40 |
| 3.1 | Activation clustering for the trigger attack | 15 |
| 3.2 | Activation clustering for the natural-feature attack | 15 |
| **Total** | | **100** |

**Submit:** this notebook with your completed code, executed outputs and figures. Complete the **YOUR WORK** cells; you may add cells if needed. Run provided cells unchanged unless explicitly marked optional and editable. All provided code is included in this notebook. Use the supplied Keras model and training settings, with a fresh model for each attack.

Use the training split to build your poisoned datasets and train your models. Keep the original clean data unchanged. Use the clean-test and attack-pool images only for evaluation; never include them in training.

## Setup: dataset and splits

We use **QMNIST**, an extension of MNIST containing handwritten digit images. The dataset is split as follows:

| Partition | Images | Role |
|---|---:|---|
| Official training partition | 60,000 | Model training and poison selection |
| First 10,000 official test images | 10,000 | Clean-test evaluation |
| Remaining official test images | 50,000 | Held-out attack-example pool |

Keep the provided ordering and splits unchanged. Attack-pool indices run from 0 to 49,999. [QMNIST dataset documentation](https://github.com/facebookresearch/qmnist).

In [ ]:
import gzip
import struct
import urllib.request
from pathlib import Path
import numpy as np

DATA = Path.cwd() / "data"
QMNIST_BASE = "https://raw.githubusercontent.com/facebookresearch/qmnist/main/"


def download(url: str, destination: Path) -> Path:
    """Download a dataset file once and reuse its local cached copy."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        tmp = destination.with_suffix(destination.suffix + ".part")
        try:
            urllib.request.urlretrieve(url, tmp)
            tmp.replace(destination)
        finally:
            tmp.unlink(missing_ok=True)
    return destination


def read_idx_gz(path: Path) -> np.ndarray:
    # Images/digit labels use unsigned bytes; extended training labels use int32.
    """Read a compressed IDX image or label file into a NumPy array."""
    with gzip.open(path, "rb") as f:
        header = f.read(4)
        if len(header) != 4 or header[:2] != b"\x00\x00":
            raise ValueError(f"Invalid IDX header: {path}")
        type_code, ndim = header[2], header[3]
        dtypes = {0x08: np.dtype("u1"), 0x0C: np.dtype(">i4")}
        if type_code not in dtypes or ndim not in (1, 2, 3):
            raise ValueError(f"Unsupported IDX type/dimensions: {path}")
        shape = struct.unpack(">" + "I" * ndim, f.read(4 * ndim))
        values = np.frombuffer(f.read(), dtype=dtypes[type_code])
    if values.size != int(np.prod(shape)):
        raise ValueError(f"Truncated IDX file: {path}")
    return values.reshape(shape).copy()


def load_qmnist_splits():
    """Return the official 60k training, 10k clean-test, and 50k attack-pool splits."""
    names = ["qmnist-train-images-idx3-ubyte.gz",
             "qmnist-train-labels-idx2-int.gz",
             "qmnist-test-images-idx3-ubyte.gz",
             "qmnist-test-labels-idx1-ubyte.gz"]
    train_images, train_metadata, test_images, test_labels = [
        read_idx_gz(download(QMNIST_BASE + name, DATA / name)) for name in names
    ]
    assert train_images.shape == test_images.shape == (60000, 28, 28)
    assert train_metadata.shape == (60000, 8)
    assert test_labels.shape == (60000,)
    train_labels = train_metadata[:, 0].astype(np.uint8)
    assert np.all(train_labels <= 9) and np.all(test_labels <= 9)
    # Preserve the ordering: first 10k for clean testing, remaining 50k for attacks.
    return (train_images, train_labels, test_images[:10000], test_labels[:10000],
            test_images[10000:], test_labels[10000:])


def normalize(images: np.ndarray) -> np.ndarray:
    """Convert uint8 pixels to float32 in [0, 1]; retain already normalized values."""
    return images.astype(np.float32) / 255.0 if images.dtype == np.uint8 else np.asarray(images, dtype=np.float32)

In [ ]:
# Load the three data splits and scale their pixels for the CNN.
SEED = 302
EPOCHS = 3
(x_train_u8, y_train, x_test_u8, y_test,
 attack_pool_u8, attack_pool_labels) = load_qmnist_splits()
x_train, x_test, attack_pool = map(normalize, (x_train_u8, x_test_u8, attack_pool_u8))
print("QMNIST training:", x_train.shape)
print("QMNIST clean test:", x_test.shape)
print("QMNIST attack pool:", attack_pool.shape)
assert len(x_train) == 60000 and len(x_test) == 10000 and len(attack_pool) == 50000
assert np.all((x_train >= 0) & (x_train <= 1))

## 0. Clean reference model

Run the supplied model and training code to establish clean accuracy on the 10,000-image QMNIST clean-test split. Use this same clean model for all before/after comparisons.

### Keras CNN implementation

The model includes a named hidden layer, `embedding`, which is available for the later activation-clustering defense. Each `train_model` call constructs and trains a fresh Keras model with the same seed.

In [ ]:
import os
# Keras 3 uses TensorFlow for this notebook. Set the backend before importing Keras.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras
from keras import layers


def build_model(seed: int = 302, learning_rate: float = 1e-3) -> keras.Model:
    """Build a fresh CNN; the named embedding layer is used later for clustering."""
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    inputs = keras.Input(shape=(28, 28, 1), name="digit_image")
    x = layers.Conv2D(16, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    embedding = layers.Dense(64, activation="relu", name="embedding")(x)
    logits = layers.Dense(10, name="logits")(embedding)
    model = keras.Model(inputs, logits, name="digit_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def train_model(images: np.ndarray, labels: np.ndarray, *, epochs: int = 3,
                batch_size: int = 256, seed: int = 302,
                learning_rate: float = 1e-3) -> tuple[keras.Model, list[float]]:
    """Train a new model from scratch and return it with the loss from each epoch."""
    model = build_model(seed=seed, learning_rate=learning_rate)
    history = model.fit(normalize(images)[..., None], np.asarray(labels),
                        batch_size=batch_size, epochs=epochs, shuffle=True, verbose=2)
    return model, history.history["loss"]


def model_output(model: keras.Model, images: np.ndarray, *, embeddings: bool = False,
                 batch_size: int = 1024) -> np.ndarray:
    """Return class scores or hidden embeddings, processing images in batches."""
    x = normalize(images)[..., None]
    if embeddings:
        extractor = keras.Model(model.inputs, model.get_layer("embedding").output)
        return extractor.predict(x, batch_size=batch_size, verbose=0)
    return model.predict(x, batch_size=batch_size, verbose=0)


def predict(model: keras.Model, images: np.ndarray) -> np.ndarray:
    """Return the most likely digit label for each image."""
    return model_output(model, images).argmax(axis=1)

In [ ]:
# Train the clean reference used for before/after comparisons.
clean_model, clean_losses = train_model(x_train, y_train, epochs=EPOCHS, seed=SEED)
print('Clean-test accuracy:', np.mean(predict(clean_model, x_test) == y_test))

## 1. Lint-trigger attack [30 points]

Create a backdoor that causes digits carrying your lint trigger to be classified as **7**, while retaining accuracy on clean digits. Complete the three parts below. Write your implementation in the **Your work** cells; run the provided cells unchanged.

| Requirement | Limit |
|---|---|
| Poisoned training examples | At most **2%**: 1,200 unique images out of 60,000 |
| Overall attack success rate (ASR) | At least **95%** |
| ASR in each evaluated source class | At least **90%** |
| Clean-accuracy drop | At most **1 percentage point** |

Your trigger should look like a small piece of natural lint or a loose fiber and keep the digit readable. Its shape, position, brightness, thickness, and opacity are your choice. Use your final trigger consistently and reproducibly in training and attack evaluation.

In [ ]:
# Fixed target digit and maximum training-poison rate for Section 1.
TARGET = 7
MAX_POISON_RATE = 0.02

### 1.1. Design the lint [5 points]

Implement **only `add_lint`**. The provided cell displays an original image beside your triggered version.

- Input: one normalized `(28, 28)` image or a batch shaped `(N, 28, 28)`.
- Output: a new `float32` array of the same shape, with finite values in `[0, 1]`.

Keep the input unchanged, and use the same lint design for individual images, batches, training, and evaluation.

In [ ]:
# YOUR WORK — implement this function.
def add_lint(images: np.ndarray) -> np.ndarray:
    """Return images with your lint, preserving their shape and the input data."""
    result = None
    return result

### Provided display

Run after completing `add_lint` to inspect the result.

In [ ]:
# Display an original image beside the image with your lint.
import matplotlib.pyplot as plt

original = x_train[0].copy()
triggered = add_lint(original.copy())
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, image, title in zip(axes, [original, triggered], ["Original", "With lint"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)

### 1.2. Poison and train [15 points]

Choose the training images to poison and train a model that meets the requirements in Section 1.

Add your lint to the selected images and assign them label 7. Keep the training set at 60,000 images, with no duplicates added. Only the selected images and labels may change; preserve the original clean arrays. Train a fresh model using the supplied architecture and training settings (`SEED`, `EPOCHS`, batch size 256, and the default optimizer and learning rate).

Fill in the following variables:

| Variable | Required value |
|---|---|
| `lint_x` | Poisoned training images: `float32`, shape `(60000, 28, 28)`, values in `[0, 1]` |
| `lint_y` | Training labels: integer array, shape `(60000,)`, values 0–9 |
| `lint_indices` | One-dimensional integer array of unique original training indices that were poisoned |
| `lint_model` | A fresh Keras model trained on your poisoned dataset using the prescribed settings |
| `poison_summary` | Dictionary described below |

Fill in `poison_summary` as follows:

| Key | What to enter |
|---|---|
| `train_size` | Total number of images in your training dataset |
| `poison_count` | Number of training images you added lint to and relabeled as 7 |
| `poison_rate` | Fraction of training images that you poisoned, between 0 and 1 |
| `source_counts` | For each original digit, how many of its training images you poisoned |

In `source_counts`, key `0` means **how many images originally labeled 0 you poisoned**; key `1` means how many images originally labeled 1 you poisoned, and so on.

Use integer counts and a fractional `poison_rate`. Count `source_counts` using the original labels; enter 0 for a digit with no poisoned examples. The ten counts must sum to `poison_count`.

In [ ]:
# YOUR WORK — provide the poisoned data, selected indices, model, and summary.
lint_x = None
lint_y = None
lint_indices = None
lint_model = None
poison_summary = {
    "train_size": None,
    "poison_count": None,
    "poison_rate": None,
    # Number of poisoned images from each original digit class.
    "source_counts": {
        0: None,
        1: None,
        2: None,
        3: None,
        4: None,
        5: None,
        6: None,
        7: None,
        8: None,
        9: None,
    },
}

### Provided gallery and summary

Run these display cells unchanged.

In [ ]:
import matplotlib.pyplot as plt

def show_gallery(images: np.ndarray, indices: np.ndarray, title: str,
                 n: int = 24, seed: int = 302,
                 labels=None, original_labels=None):
    """Show a reproducible sample of images, optionally displaying label changes."""
    indices = np.asarray(indices)
    rng = np.random.default_rng(seed)
    chosen = rng.choice(indices, min(n, len(indices)), replace=False) if len(indices) else []
    fig, axes = plt.subplots(3, 8, figsize=(12, 5))
    for ax, idx in zip(axes.flat, chosen):
        ax.imshow(images[idx], cmap="gray", vmin=0, vmax=1)
        caption = str(idx)
        if labels is not None and original_labels is not None:
            caption += f": {original_labels[idx]} -> {labels[idx]}"
        ax.set_title(caption, fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.close(fig)  # display(fig) in a notebook should render it only once
    return fig

In [ ]:
# PROVIDED DISPLAY — run unchanged; show the summary and sample poisons.
print(f"Poisoned: {poison_summary['poison_count']}/{poison_summary['train_size']} ({poison_summary['poison_rate']:.2%})")
print("Original digit | Poisoned training images")
for digit, count in poison_summary["source_counts"].items():
    print(f"{digit:>14} | {count}")
display(show_gallery(lint_x, lint_indices, "Lint poisons (index: original -> poisoned label)",
                     labels=lint_y, original_labels=y_train))

### 1.3. Launch the attack and report [10 points]

For evaluation, use the **provided attack set**: 4,500 held-out clean images, with 500 from each non-7 digit class. `attack_images` contains the images and `attack_source_labels` their true labels. Do not change the selection, order, or true labels. The target is 7 throughout this task.

Submit:

- `triggered_attack_images`: your final lint applied to the provided attack images, with the same shape, order, and normalized image format.
- `lint_report`: a dictionary of your calculated results, using the exact keys below. The provided report cell prints it; do not write formatting code.

| Required keys | Type / meaning |
|---|---|
| `clean_count` | Integer: number of clean-test images evaluated |
| `clean_accuracy_before`, `clean_accuracy_after` | Fractions: accuracy of the clean and poisoned models on the untouched clean test |
| `utility_drop_pp` | Clean-accuracy drop in percentage points; positive means accuracy decreased |
| `train_size`, `poison_count` | Integers: training-set size and number of poisoned training examples |
| `poison_rate` | Fraction: training poisoning rate |
| `attack_count`, `attack_successes` | Integers: evaluated attack attempts and successful misclassifications as the target |
| `overall_asr` | Fraction: attack success rate across all 4,500 provided images |
| `clean_model_asr` | Fraction: ASR of the original, unpoisoned model on the same lint-triggered images |
| `per_class` | List of nine dictionaries, one for each original digit except 7 |

Each `per_class` dictionary must contain `source`, `count`, and `attack_successes` as integers and `asr` as a fraction. `count` is the number of its attack images evaluated. `attack_successes` is the number of triggered images from that original digit that the poisoned model classified as 7.

### Provided attack set

Run unchanged. The images are clean; launching your attack is your work.

In [ ]:
# PROVIDED ATTACK SET — run unchanged; selection is independent of predictions.
ATTACK_SAMPLE_SEED = 1302
ATTACK_PER_CLASS = 500
rng = np.random.default_rng(ATTACK_SAMPLE_SEED)
attack_indices = np.sort(np.concatenate([
    rng.choice(np.flatnonzero(attack_pool_labels == c), ATTACK_PER_CLASS, replace=False)
    for c in range(10) if c != TARGET
]))
attack_images = attack_pool[attack_indices].copy()
attack_source_labels = attack_pool_labels[attack_indices].copy()
print(f"Attack set: {len(attack_images)} clean images; {ATTACK_PER_CLASS} per non-7 class.")

In [ ]:
# YOUR WORK — provide the triggered attack images and calculated report.
triggered_attack_images = None
lint_report = {
    "clean_count": None,
    "clean_accuracy_before": None,
    "clean_accuracy_after": None,
    "utility_drop_pp": None,
    "train_size": None,
    "poison_count": None,
    "poison_rate": None,
    "attack_count": None,
    "attack_successes": None,
    "overall_asr": None,
    "clean_model_asr": None,
    # Results for each original digit in the provided attack set.
    "per_class": [
        {"source": 0, "count": None, "attack_successes": None, "asr": None},
        {"source": 1, "count": None, "attack_successes": None, "asr": None},
        {"source": 2, "count": None, "attack_successes": None, "asr": None},
        {"source": 3, "count": None, "attack_successes": None, "asr": None},
        {"source": 4, "count": None, "attack_successes": None, "asr": None},
        {"source": 5, "count": None, "attack_successes": None, "asr": None},
        {"source": 6, "count": None, "attack_successes": None, "asr": None},
        {"source": 8, "count": None, "attack_successes": None, "asr": None},
        {"source": 9, "count": None, "attack_successes": None, "asr": None},
    ],
}

### Provided report format

Run unchanged. This displays your calculated values in the required order.

In [ ]:
# PROVIDED DISPLAY — run unchanged. No metrics are calculated here.
def display_attack_report(r):
    """Print the supplied metrics in a fixed format without recalculating them."""
    print(f"Clean accuracy: {r['clean_accuracy_before']:.2%} -> {r['clean_accuracy_after']:.2%} "
          f"(drop {r['utility_drop_pp']:+.2f} pp; n={r['clean_count']})")
    print(f"Poisoned: {r['poison_count']}/{r['train_size']} ({r['poison_rate']:.2%})")
    print(f"Attack success: {r['attack_successes']}/{r['attack_count']} ({r['overall_asr']:.2%})")
    print(f"Clean-model ASR on lint-triggered images: {r['clean_model_asr']:.2%}")
    print("\nSource      N  Successes      ASR")
    for row in sorted(r["per_class"], key=lambda row: row["source"]):
        print(f"{row['source']:>6} {row['count']:>6} {row['attack_successes']:>10} {row['asr']:>8.2%}")

In [ ]:
# PROVIDED REPORT — run unchanged; display your completed results.
display_attack_report(lint_report)

## 2. Natural-feature attack: 1s with hats [40 points]

Exploit a feature already present in some handwritten 1s: a small cap or angled stroke near the top, called a **hat** here. Your goal is to make these 1s be classified as **7** without adding any pixels to them.

First, identify hat-shaped 1s in the training dataset and choose which examples to poison. Then change their labels to 7, train a fresh model, and evaluate the attack on the provided held-out examples. Finding a training selection that makes the attack succeed is part of your task.

You may start with the provided K-means code to organize training images for inspection, or use another method of your choice. You may modify, replace, or remove that code. Whatever method you use, visually check your selected images and explain how you found them.

You decide which training examples to use and how many. **There is no prescribed poison count or poisoning-rate limit for this task.** Report your actual selection, attack success, and utility drop.

**Achieve at least 50% ASR on the provided 100-image attack set. There is no clean-accuracy-drop limit.**

### 2.1. Identify training hats [15 points]

Discover training images labeled **1** that contain a cap or angled entry stroke near the top. Select the examples yourself; no training-hat set or required count is supplied.

Produce `hat_train_indices`: a one-dimensional integer array of unique, original indices into `x_train`, all with original label 1. Show your selected examples using the provided display.

**Explain your visual criterion, discovery method, and exclusions in concise code comments. If your code contains manually chosen indices, explain how you obtained them.**

The optional clustering helper below is a starting point. You may **modify, replace, or remove it**, including using a different discovery method.

### Optional editable helper

`cluster_ones(images, labels, ...)` groups digit-1 writing styles. `show_ones_groups(groups, images, cluster_ids=None)` displays galleries. You may change or delete this cell.

In [ ]:
# OPTIONAL STARTING CODE — modify, replace, or remove as needed.
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

def cluster_ones(images, labels, n_clusters=12, n_components=30, seed=302):
    """Group digit-1 writing styles by upper-image appearance; no hat labels are inferred."""
    positions = np.flatnonzero(labels == 1)
    pixels = normalize(images[positions])[:, 2:18, 2:24].reshape(len(positions), -1)
    features = PCA(n_components=n_components, random_state=seed).fit_transform(pixels)
    assignments = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(features)
    return {"positions": positions, "assignments": assignments,
            "counts": np.bincount(assignments, minlength=n_clusters), "n_clusters": n_clusters}


def show_ones_groups(groups, images, cluster_ids=None):
    """Display candidate groups for visual review, without deciding which images are hats."""
    if cluster_ids is None:
        cluster_ids = range(groups["n_clusters"])
    for cluster_id in cluster_ids:
        members = groups["positions"][groups["assignments"] == cluster_id]
        display(show_gallery(images, members, f"Digit-1 group {cluster_id}: {len(members)} images", n=16))

In [ ]:
# YOUR WORK — discover and record training hats; explain your method and choices here.
hat_train_indices = None

In [ ]:
# Display your selection without changing its pixels or labels.
print("Selected training hats:", len(hat_train_indices))
display(show_gallery(x_train, hat_train_indices, "Selected training hats"))

### 2.2. Poison and train [10 points]

Use `hat_train_indices` from 2.1 to create a poisoned training dataset. Change only those images' labels from 1 to 7. Keep every image pixel-for-pixel unchanged, retain all 60,000 examples, and leave every other label unchanged. Preserve the original clean arrays.

| Variable | Required value |
|---|---|
| `natural_x` | Training images: `float32`, shape `(60000, 28, 28)`, values in `[0, 1]` |
| `natural_y` | Integer training labels, shape `(60000,)`, values 0–9 |
| `natural_model` | Fresh Keras model trained using the supplied architecture and settings |
| `natural_poison_summary` | Dictionary with integer `train_size`, integer `poison_count`, and fractional `poison_rate` |

You choose the number of training hats. There is no poison-rate cap for this section.

Train a fresh model using the same supplied architecture and training settings as in Section 1. Use integer `train_size` and `poison_count` values and a fractional `poison_rate` in `natural_poison_summary`.

In [ ]:
# YOUR WORK — construct the natural-feature poisoned dataset and train a fresh model.
natural_x = None
natural_y = None
natural_model = None
natural_poison_summary = {
    "train_size": None,
    "poison_count": None,
    "poison_rate": None,
}

In [ ]:
# Print your summary and show the recorded label changes.
print(f"Poisoned: {natural_poison_summary['poison_count']}/{natural_poison_summary['train_size']} "
      f"({natural_poison_summary['poison_rate']:.2%})")
display(show_gallery(natural_x, hat_train_indices, "Natural-feature poisons",
                     labels=natural_y, original_labels=y_train))

### 2.3. Launch the attack and report [15 points]

Evaluate the poisoned model on **all 100 provided hat-shaped 1s**, unchanged. Achieve **ASR ≥50%**: at least 50 must be predicted as 7.

Evaluate the same hats with the original clean model as a baseline, and evaluate both models on the untouched clean-test split. Keep the provided evaluation images and indices unchanged. Fill in the `natural_report` skeleton using the exact fields below; the provided code prints the results.

| Required fields | Meaning |
|---|---|
| `clean_count` | Number of untouched clean-test images evaluated: 10,000 |
| `clean_accuracy_before`, `clean_accuracy_after` | Clean and poisoned models' accuracies on the same clean-test split |
| `utility_drop_pp` | Clean-accuracy drop in percentage points; positive means accuracy decreased |
| `train_size`, `poison_count`, `poison_rate` | Training size, number of poisons, and fractional poisoning rate |
| `hat_count`, `hat_successes`, `hat_asr` | Hat evaluation size (100), predictions of 7, and ASR |
| `clean_model_asr` | ASR of the original, unpoisoned model on the same 100 hats |

Use integer counts and fractional accuracy, ASR, and poison-rate values in `[0, 1]`. `utility_drop_pp` is in percentage points and may be negative if accuracy improves. Report the actual clean-accuracy change; there is no clean-accuracy-drop limit for this attack.

In [ ]:
# PROVIDED EVALUATION SET — run unchanged.
# Fixed visual annotations of held-out hats.
# Indices are zero-based attack_pool positions; original QMNIST test row = 10000 + index.
hat_attack_indices = np.array([
    536, 1537, 2398, 3712, 4126, 4277, 4677, 4705, 6767, 7838, 7930, 8165, 9104, 9366, 9471, 9566, 9626,
    9999, 10405, 10668, 11191, 11714, 12896, 14210, 14391, 15039, 15554, 15647, 16398, 17205, 17452,
    17899, 18017, 18717, 19157, 19591, 19836, 19898, 20326, 20923, 20926, 22085, 22677, 22889, 23979,
    25742, 25829, 27384, 27637, 28567, 28712, 29077, 29205, 29494, 29904, 30248, 30502, 31779, 32000,
    32095, 32326, 33137, 33527, 34006, 34027, 34742, 35012, 36776, 37088, 37270, 37343, 37892, 38110,
    40293, 40598, 40787, 40936, 41081, 41644, 41900, 41936, 42736, 43224, 43259, 43627, 43901, 43928,
    44719, 44783, 44795, 45249, 45331, 45828, 45888, 47035, 47097, 48126, 49367, 49458, 49974
], dtype=int)

hat_images = attack_pool[hat_attack_indices].copy()
for start in range(0, 100, 24):
    display(show_gallery(attack_pool, hat_attack_indices[start:start + 24],
                         f"Fixed evaluation hats {start + 1}–{min(start + 24, 100)}"))

In [ ]:
# YOUR WORK — evaluate the models and calculate the required report values.
natural_report = {
    "clean_count": None,
    "clean_accuracy_before": None,
    "clean_accuracy_after": None,
    "utility_drop_pp": None,
    "train_size": None,
    "poison_count": None,
    "poison_rate": None,
    "hat_count": None,
    "hat_successes": None,
    "hat_asr": None,
    "clean_model_asr": None,
}

In [ ]:
# PROVIDED REPORT — print calculated values without deriving metrics.
def display_natural_report(r):
    """Print clean utility, poison rate, and target behavior on hats."""
    print(f"Clean accuracy: {r['clean_accuracy_before']:.2%} -> {r['clean_accuracy_after']:.2%} "
          f"(drop {r['utility_drop_pp']:+.2f} pp; n={r['clean_count']})")
    print(f"Poisoned: {r['poison_count']}/{r['train_size']} ({r['poison_rate']:.2%})")
    print(f"Hat 1s: {r['hat_successes']}/{r['hat_count']} -> 7 ({r['hat_asr']:.2%}); "
          f"clean-model ASR: {r['clean_model_asr']:.2%}")

display_natural_report(natural_report)

## 3. Defense: activation clustering [30 points]

Apply activation clustering to both attacks: the synthetic trigger from Section 1 and the natural feature from Section 2. The clustering and display functions are provided; choose one configuration, inspect its clusters, select suspicious clusters, and calculate detection metrics. For each attack, show one chosen configuration and galleries from every cluster, then select suspicious clusters and report detection results.

**Why activations?** A hidden layer represents each image as a vector of learned features. Poisoned examples labeled 7 may produce different feature patterns from genuine 7s, even though both have the same training label. Clustering these vectors can expose groups worth inspecting.

The **penultimate layer** is the last hidden layer immediately before the output layer. It is a useful starting point because it captures features used for classification before they are compressed into class scores. Earlier layers describe different kinds of features, so a different layer may separate the poisons better. There is no guaranteed best layer.

**Methodology:** cluster the activations of training images **labeled 7**, inspect cluster sizes, and visually examine the groups for anomalies. Small clusters are potential candidates for suspicion. Confirm your suspicions from the images rather than treating size as proof. More than one cluster may be suspicious.

**For each attack, achieve F1 ≥ 80% (a fractional `f1` value of at least `0.80`) within training images labeled 7.**

Choose configurations and suspicious clusters from the images and cluster sizes, before consulting the recorded poison indices for scoring. Submit the configuration you chose; a comparison of multiple configurations and written interpretations are not required.

### 3.1. Defend against the synthetic trigger [15 points]

Use `lint_model`, `lint_x`, and `lint_y` from Section 1. Complete the two parts below.

#### 3.1.1. Apply activation clustering and show all clusters [5 points]

Apply `activation_clusters` to `lint_model`, `lint_x`, and `lint_y`. Choose the activation layer (`layer_name`), number of PCA dimensions (`n_components`), and number of clusters (`n_clusters`). You may adjust these settings until the images reveal a group you suspect contains poisoned examples.

Save **one chosen configuration's result** as `lint_clusters`. Run the provided display to print its settings and cluster sizes, then show a gallery from **every cluster**.

In [ ]:
# PROVIDED CLUSTERING — run unchanged.
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


def activation_clusters(model, images, labels, *, layer_name=None, class_label=7,
                        n_components=10, n_clusters=2, seed=302):
    """Extract a selected layer's features, apply PCA, and cluster one training label."""
    positions = np.flatnonzero(labels == class_label)
    layer_name = layer_name or model.layers[-2].name  # Penultimate layer by default.
    extractor = keras.Model(model.inputs, model.get_layer(layer_name).output)
    activations = extractor.predict(normalize(images[positions])[..., None], batch_size=512, verbose=0)
    activations = activations.reshape(len(positions), -1)  # Also support convolutional layers.
    if not 1 <= n_components <= min(activations.shape):
        raise ValueError("PCA dimensions must fit the number of examples and layer features")
    if not 2 <= n_clusters < len(positions):
        raise ValueError("Choose at least two clusters and fewer clusters than examples")
    reduced = PCA(n_components=n_components, random_state=seed).fit_transform(activations)
    assignments = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(reduced)
    counts = np.bincount(assignments, minlength=n_clusters)
    return {"layer_name": layer_name, "n_components": n_components, "n_clusters": n_clusters,
            "positions": positions, "assignments": assignments, "counts": counts,
            "reduced": reduced}


def display_clustering_config(result):
    """Print one chosen configuration and its cluster sizes."""
    print(f"Layer: {result['layer_name']}  PCA: {result['n_components']}  "
          f"K: {result['n_clusters']}  Sizes: {result['counts'].tolist()}")


def show_activation_clusters(result, images, cluster_ids=None):
    """Show chosen activation clusters, using original training indices for the images."""
    if cluster_ids is None:
        cluster_ids = range(result["n_clusters"])
    for cluster_id in cluster_ids:
        members = result["positions"][result["assignments"] == cluster_id]
        display(show_gallery(images, members,
                             f"Cluster {cluster_id}: {len(members)} images", n=16))

In [ ]:
# List hidden layers.
for layer in lint_model.layers[1:-1]:
    print(layer.name, tuple(layer.output.shape[1:]))

In [ ]:
# YOUR WORK — apply activation clustering with your chosen configuration.
lint_clusters = None

In [ ]:
# Print the chosen settings and show every cluster, without scoring detection.
display_clustering_config(lint_clusters)
show_activation_clusters(lint_clusters, lint_x)

#### 3.1.2. Show suspicious clusters and evaluate [10 points]

Select the suspicious clusters from 3.1.1 and record their IDs in `suspicious_clusters`.

Evaluate how successfully the defense identifies poisoned training data by calculating **true positives, false positives, false negatives, precision, recall, and F1**. Enter your results in the `lint_defense_report` skeleton. Evaluate only training images labeled 7: your selected clusters identify detected poisons, while `lint_indices` records the actual poisons.

You may use scikit-learn's [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) for the counts and [`precision_recall_fscore_support`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html) for the metrics.

**Required result: F1 ≥80%.**

Use integers for `tp`, `fp`, and `fn`, and fractions for `precision`, `recall`, and `f1`; use 0 for a metric with a zero denominator. Run the provided display to show only the selected clusters and print the report. No written interpretation is required.

In [ ]:
# YOUR WORK — select suspicious clusters, then calculate the report values.
suspicious_clusters = None
lint_defense_report = {
    "tp": None,
    "fp": None,
    "fn": None,
    "precision": None,
    "recall": None,
    "f1": None,
}

In [ ]:
# Display only your selected groups and your calculated detection report.
def display_defense_report(r):
    """Print submitted detection counts and metrics without calculating them."""
    print(f"TP: {r['tp']}  FP: {r['fp']}  FN: {r['fn']}")
    print(f"Precision: {r['precision']:.2%}  Recall: {r['recall']:.2%}  F1: {r['f1']:.2%}")

print("Suspicious clusters:", suspicious_clusters)
show_activation_clusters(lint_clusters, lint_x, suspicious_clusters)
display_defense_report(lint_defense_report)

### 3.2. Defend against the natural feature [15 points]

Use `natural_model`, `natural_x`, and `natural_y` from Section 2. Complete the two parts below.

#### 3.2.1. Apply activation clustering and show all clusters [5 points]

Apply `activation_clusters` to `natural_model`, `natural_x`, and `natural_y`. Choose the activation layer (`layer_name`), number of PCA dimensions (`n_components`), and number of clusters (`n_clusters`). You may adjust these settings until the images reveal a group you suspect contains poisoned examples.

Save **one chosen configuration's result** as `natural_clusters`. Run the provided display to print its settings and cluster sizes, then show a gallery from **every cluster**.

In [ ]:
# List hidden layers for the natural-feature model.
for layer in natural_model.layers[1:-1]:
    print(layer.name, tuple(layer.output.shape[1:]))

In [ ]:
# YOUR WORK — apply activation clustering to the natural-feature poisoned data.
natural_clusters = None

In [ ]:
# Print the chosen settings and show every cluster, without scoring detection.
display_clustering_config(natural_clusters)
show_activation_clusters(natural_clusters, natural_x)

#### 3.2.2. Show suspicious clusters and evaluate [10 points]

Select the suspicious clusters from 3.2.1 and record their IDs in `natural_suspicious`.

Evaluate how successfully the defense identifies poisoned training data by calculating **true positives, false positives, false negatives, precision, recall, and F1**. Enter your results in the `natural_defense_report` skeleton. Evaluate only training images labeled 7: your selected clusters identify detected poisons, while `hat_train_indices` records the actual poisons.

You may use scikit-learn's [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) for the counts and [`precision_recall_fscore_support`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html) for the metrics.

**Required result: F1 ≥80%.**

Use integers for `tp`, `fp`, and `fn`, and fractions for `precision`, `recall`, and `f1`; use 0 for a metric with a zero denominator. Run the provided display to show only the selected clusters and print the report. No written interpretation is required.

In [ ]:
# YOUR WORK — select suspicious clusters, then calculate the report values.
natural_suspicious = None
natural_defense_report = {
    "tp": None,
    "fp": None,
    "fn": None,
    "precision": None,
    "recall": None,
    "f1": None,
}

In [ ]:
# Show only the selected natural-feature examples, then display your calculated metrics.
print("Suspicious clusters:", natural_suspicious)
show_activation_clusters(natural_clusters, natural_x, natural_suspicious)
display_defense_report(natural_defense_report)